## B2 - Image Capture and Data Collection
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 17th, 2026

About: In this notebook you'll learn how to capture and store images from your camera and how this can be done repeatedly to collect a dataset.

### This is the start of the machine learning portion of our course!

<span style="color: green; font-size: 55px; font-style: italic;">Student Discussion Time</span>


## **STUDENT QUESTIONS**:
Talk these over with your group.

 + **Data vs data points:** What is a data? What are data points?

 + **Images:** We just learned how to collect images, can those be used as data points? What can the images tell us about the world? 


<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY B2.1: Saving Captured Images</span>


### In the last notebook we followed a new template for collecting images and displaying them. Now we'll add a new element, saving those images to the file system on the robot.
Step 1. Import libraries designed specifically for the camera, camera data, and visualizing the camera data (pictures).  
Step 2. Create a camera object that can be called to capture images.  
Step 3. Create jupyter laboratory widgets to visualize the camera data.  
Step 4. Link your camera object and the widget so that new images are automatically streamed.  
Step 5. Display the widgets. 

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# This cell imports all the libraries we'll use for this notebook

# Importing graphical user interface libraries
import traitlets
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual, Layout
from IPython.display import display

# Importing file and data management libraries
import os
from uuid import uuid1
# Importing custom helper tools
from jupyter_utils import register_click_handler, register_dlink
from capture_utils import ensure_directory, save_image, count_images

# Step 1. Import our custom library for our TraitletCamera
from jetcam_lite import TraitletCamera, bgr8_to_jpeg

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####

### Camera Setup ###
# Step 2. Instantiate and start the camera
camera = TraitletCamera()
camera.start()

# Step 3. Create the widget to display the image feed, camera image is 2x for better visualization
image_widget = widgets.Image(format='jpeg', width=camera.width, height=camera.height)

# Step 4.  Link the widget and the camera feed, then display
# register_dlink (instead of calling traitlets.dlink() directly) keeps this cell
# safe to re-run: it won't stack a second live link onto the widget.
register_dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)
# Display the image widget
display(image_widget)

### Saving images to disk
When saving images to disk we need two things:
 + A name for the file
 + A location where we will save the file

**About the name**: we could just chose picture.jpg, but if we run the program again the original will be replaced with the new image. This isn't what we want when we're trying to build a dataset of thousands of images.  
  
  So we need to give each file a unique name, generated as a combination of useful descriptors, like "blocks," and a semi-random string that guarantees that no two photos we take will have the same name. The ```uuid``` library is used to generate a unique string or "Universally Unique IDentifier". More on this soon.

  **About the location**: normally we can store our images wherever we like, but since we're about to start building datasets, the location where we store images is **very** important.  
  Imagine we're teaching our AI how to recognize dogs from cats, we'd want only pictures of dogs in the dogs folder, and oonly pictures of cats in the cats folder. Otherwise, the AI might be easily confused and produce inaccurate results.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Testing the universally unique identifier (UUID) library
# This cell creates a UUID then prints it. Feel free to run this cell several times to prove to yourself that the ID never repeats.
print(uuid1())

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Using the python os library, we can check the contents of a directory and create new directories (folders).

# We can use the library to get our currect working directory and list its contents.

# Get the current working directory
cwd = os.getcwd()

# Print the directory path
print("Current Working Directory:", cwd)

# List files and folders in the current working directory
contents = os.listdir('.')
print(contents)

# We can also create new directories. Here, we're going to create a new directory called "test"
# After running this cell look in the file exploreer to the left for your new directory.
dirExists = os.path.exists("test")
if not dirExists: 
    os.makedirs("test")
else: print("Directory exists aleardy.")

# NOTE: you might need to refresh your file explorer pane to see the new directory

### Putting it all together
We're now going to save the current frame from our live vidoe feed to our new "test" directory.  
To do this, we'll compose a file name with a path to the directory as a string using uuid.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Capturing an image and saving it with a unique name to the test directory.

# Create a new filename with the coordinates appended, then save the file in the new directory.
uuid = 'robotics_training_institute_%s' % uuid1() #uuid(1) creates a unique string each time it's called
image_path = os.path.join('test/', uuid + '.jpg') # Building the path & filename for the new labeled data point
print(image_path)

# Save the file to disk with the new name.
with open(image_path, 'wb') as f:
    f.write(image_widget.value) # camera.value for csi camera, image.value for jetcam

### You can now look instide your test directory for the image you took!

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY B2.2: Creating Your First Dataset</span>


### Our Goal: Create an AI that can tell the difference between red and blue blocks

To achieve our goal, in this next activity we'll collect images of red blocks and blue blocks.  
We'll store these images in new 'red' and 'blue' directories.  
To do this, we'll create a simple graphical user interface event-driven programming.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Creating the user interface for our data collection
# Define a consistent layout
button_layout = Layout(width='200px')
text_layout = Layout(width='200px')

# Two buttons, one for capturing and saving images of red blocks and the other for blue blocks.
blue_button = widgets.Button(description='Collect BLUE', layout=button_layout)
blue_button.style.button_color = '#007BFF' 

red_button = widgets.Button(description="Collect RED", layout=button_layout)
red_button.style.button_color = '#FF0000'

# Text boxes to display the number of files in each directory
red_count = widgets.Text(description='Red:', disabled=False, layout=text_layout)
blue_count = widgets.Text(description='Blue:', disabled=False, layout=text_layout)

display(image_widget)
display(widgets.HBox([blue_button, red_button]))
display(widgets.HBox([blue_count, red_count]))

### Right now the bottons don't work. We'll next add the callback functions that save images to their respective folders  
In the following code cell, I create the folders that will hold the photos of red blocks and of blue blocks. These directories are in the ```/Datasets/red_blue_dataset/``` directory.  

Then I create two callback functions that use the save image pattern we learned in the last activity. These functions save the images in the respective folder.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Creating button callback functions and the directories where we'll save the images

# Creating a directory for our dataset and the individual directories for each type of image: red and blue
# os.path.expanduser("~") resolves to the current user's home directory, so this
# notebook works correctly on any robot in the fleet, regardless of username.
red_blue_dataset = os.path.join(os.path.expanduser("~"), "Datasets", "red_blue_dataset")  # The name and location of our new directory
blue_classification_directory = os.path.join(red_blue_dataset, "blue")  # Directory for the blue images
red_classification_directory = os.path.join(red_blue_dataset, "red")   # Directory for the red images

# ensure_directory() creates each folder if it doesn't already exist, and does
# nothing (no error) if it does -- safe to re-run this cell any number of times.
ensure_directory(blue_classification_directory)
ensure_directory(red_classification_directory)

# Initialize the counters from whatever is already saved on disk, so re-opening
# this notebook mid-collection shows accurate counts right away.
blue_count.value = str(count_images(blue_classification_directory))
red_count.value = str(count_images(red_classification_directory))


# This function runs every time the BLUE button is clicked.
def blue_capture_clicked(button):
    # save_image() builds a unique filename, writes the file, and returns the path used.
    save_image(image_widget.value, blue_classification_directory, label='blue')
    blue_count.value = str(count_images(blue_classification_directory))


# This function runs every time the RED button is clicked.
def red_capture_clicked(button):
    save_image(image_widget.value, red_classification_directory, label='red')
    red_count.value = str(count_images(red_classification_directory))


# Connect the buttons to their respective functions from above. When the button is clicked the function is executed.
# register_click_handler (instead of calling .on_click() directly) keeps this cell
# safe to re-run: it won't stack a second copy of the handler onto the button,
# which would otherwise save two (or three, or four...) images per click.
register_click_handler(blue_button, blue_capture_clicked)
register_click_handler(red_button, red_capture_clicked)

### Now we have created the user interface and setup the back-end functions for event-driven programming.
### Your job is to collect images of red and blue blocks individually

When collecting data for a dataset like this, it is important to have variety in your dataset.  
Variety can look like:  
 + Different orientations of the block
 + Different backgrounds behind the block
 + Different lighting conditions on the block
 + Different distances to the block. 

### Playing with all these ways to add variety to the data, now collect 200 images of blue blocks and 200 images of red blocks.
Run the next cell for your data collection user interface.

In [ ]:
display(image_widget)
display(widgets.HBox([blue_button, red_button]))
display(widgets.HBox([blue_count, red_count]))

### Once you have 200 images in each folder move to the next notebook B3a_Machine_Learning.